In [2]:
import cv2
import os
from ultralytics import YOLO

# 1. Hazır eğitilmiş segmentasyon modelini yüklüyoruz
model = YOLO("yolov8n-seg.pt")

# 2. Videoyu İndirilenler klasöründen çekiyoruz
user_home = os.path.expanduser("~")
video_path = os.path.join(user_home, "Downloads", "gsmanu.mp4")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("HATA: 'gsmanu.mp4' dosyası İndirilenler klasöründe bulunamadı!")
    print("Lütfen dosyanın İndirilenler klasöründe ve adının doğru olduğundan emin olun.")
else:
    # Çıktı videosunun kaydedileceği yer
    output_path = os.path.join(user_home, "Downloads", "gsmanu_proje_cikti.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, 30.0, (int(cap.get(3)), int(cap.get(4))))

    print("GS - MANU videosu için yapay zeka segmentasyonu başladı... Lütfen bekleyin.")

    frame_count = 0
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
            
        frame_count += 1
        
        # Sadece insan (oyuncu, hakem vb.) sınıfını filtrele
        results = model(frame, classes=[0], verbose=False) 
        
        oyuncu_sayisi = len(results[0].boxes) if results[0].boxes is not None else 0
        annotated_frame = results[0].plot()
        
        # --- KARAR DESTEK SİSTEMİ MANTIĞI ---
        if oyuncu_sayisi > 8:
            karar_mesaji = "Karar Destek: Yogun bolge presi! Pas mesafesini uzatin."
            renk = (0, 0, 255) # Kırmızı
        elif oyuncu_sayisi > 3:
            karar_mesaji = "Karar Destek: Dengeli dagilim. Kanat hucumunu deneyin."
            renk = (0, 255, 0) # Yeşil
        else:
            karar_mesaji = "Karar Destek: Savunma az azinlikta! Hizli hucuma cikin."
            renk = (255, 0, 0) # Mavi
            
        cv2.putText(annotated_frame, karar_mesaji, (30, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, renk, 2, cv2.LINE_AA)
        
        cv2.putText(annotated_frame, f"Tespit Edilen Oyuncu: {oyuncu_sayisi}", (30, 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        
        out.write(annotated_frame)

    cap.release()
    out.release()
    print("\n" + "="*50)
    print("İŞLEM BAŞARIYLA TAMAMLANDI!")
    print(f"Yeni çıktı videosu şuraya kaydedildi:\n{output_path}")
    print("="*50)

GS - MANU videosu için yapay zeka segmentasyonu başladı... Lütfen bekleyin.

İŞLEM BAŞARIYLA TAMAMLANDI!
Yeni çıktı videosu şuraya kaydedildi:
C:\Users\cagin\Downloads\gsmanu_proje_cikti.mp4
